# Power / Minimum-Detectable-Effect Analysis for RQ1 -- Bio-Specific Elite List

**Owner:** Jici · **Depends on:** `rq1_methodology.md` §10 (subject-specific
robustness check, by Yan-Bo), Option B bio-specific labels

**This is a parallel run of `power_analysis.ipynb`**, substituting the
QS-overall elite list with the **QS World University Rankings by Subject
2026: Biological Sciences** list used in the RQ1 robustness check. Per team
convention (see `rq1_methodology.md` §10), this is a fully parallel
pipeline -- it does not touch or overwrite the original QS-overall power
analysis (`power_analysis.ipynb` / `results/rq1_power_analysis.json`).

**Why redo the power analysis here:** the robustness check (§10) showed the
bio-specific SPD point estimate (+0.031) is close to, but not identical to,
the QS-overall estimate (+0.029), and the labeled-slot count differs slightly
(591 vs 590, 112 vs 102 elite). Since MDE depends on both the baseline rate
and the exact sample composition, the QS-overall power curve does not
automatically transfer -- this notebook re-derives it under the bio-specific
design to confirm the power/MDE conclusion is *also* robust to the
elite-list choice, not just the point estimate and CI.

**Method:** identical to `power_analysis.ipynb` §0-8 (see that notebook for
the full methodology writeup and disclosed limitations) -- only the baseline
share, label files, and observed SPD are swapped for the bio-specific values.

In [ ]:
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt


## 0. Config (bio-specific)

In [ ]:
BASELINE_SHARE = 0.1584     # bio-specific Option B baseline (70/442 found)
N_BOOT_VALIDATE = 5000      # matches rq1_methodology.md §10's bio bootstrap
N_BOOT_POWER    = 3000      # inner bootstrap reps per simulated dataset
R_REPLICATES    = 2000      # outer simulation replicates per target SPD
TARGET_SPDS     = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15]
OBSERVED_SPD    = 0.0311    # bio-specific RQ1 robustness-check point estimate
POWER_TARGET    = 0.80
SAMPLE_LABELS_FILE    = 'sample_labels_1000_bio.json'
RETRIEVAL_LABELS_FILE = 'retrieval_labels_bio.json'

def find(hint):
    for base in ['data', '.', '..']:
        hits = glob.glob(f'{base}/**/{hint}', recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(hint)


## 1. Reconstruct real per-query (n_q, e_q) -- bio-specific labels

Same neutral-query scope (q001-q100) and same reconstruction logic as the
QS-overall notebook, but reading `sample_labels_1000_bio.json` +
`retrieval_labels_bio.json` (elite defined by the QS Biological Sciences
Top-50, not the overall ranking).

In [ ]:
with open(find('retrieval_results.json')) as f:
    retrieval = json.load(f)
queries = [q for q in retrieval if q.get('type', 'neutral') == 'neutral']
queries = [q for q in queries if q['query_id'] <= 'q100']
print(f'neutral queries: {len(queries)}')

label_of = {}
for fn in [SAMPLE_LABELS_FILE, RETRIEVAL_LABELS_FILE]:
    with open(find(fn)) as f:
        for r in json.load(f)['labels']:
            label_of[str(r['paper_id'])] = r.get('elite_label')

n_list, e_list = [], []
for q in queries:
    labs = [label_of.get(pid) for pid in q['retrieved_paper_ids']]
    labs = [x for x in labs if x is not None]
    n_list.append(len(labs))
    e_list.append(sum(labs))
n_arr = np.array(n_list)
e_arr_real = np.array(e_list)
Q = len(n_arr)
print(f'total labeled slots: {n_arr.sum()} | total elite: {e_arr_real.sum()} | '
      f'retrieved elite share: {e_arr_real.sum()/n_arr.sum():.4f}')

## 2. Validate: reproduce the bio-specific RQ1 result

Must match `results/rq1_optionB_result_bio.json` (591 labeled slots, 112
elite, SPD +0.0311, CI [-0.0050, +0.0687]) before trusting anything
downstream.

In [ ]:
def bootstrap_ci(n_arr, e_arr, baseline, n_boot, rng):
    Qn = len(n_arr)
    idx = rng.integers(0, Qn, size=(n_boot, Qn))
    n_tot = n_arr[idx].sum(axis=1)
    e_tot = e_arr[idx].sum(axis=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        spds = e_tot / n_tot - baseline
    return np.nanpercentile(spds, [2.5, 97.5])

rng = np.random.default_rng(42)
spd_point = e_arr_real.sum() / n_arr.sum() - BASELINE_SHARE
ci_lo, ci_hi = bootstrap_ci(n_arr, e_arr_real, BASELINE_SHARE, N_BOOT_VALIDATE, rng)
print(f'SPD point estimate : {spd_point:+.4f}   (bio robustness check reports +0.0311)')
print(f'Bootstrap 95% CI   : [{ci_lo:+.4f}, {ci_hi:+.4f}]   (reports [-0.0050, +0.0687])')
assert abs(spd_point - 0.0311) < 0.001, 'SPD point estimate does not match bio RQ1 result -- check labels'
print('\nMATCH CONFIRMED.')

## 3. Power simulation (bio-specific)

In [ ]:
def power_at(target_spd, n_arr, baseline, R, n_boot, rng):
    Qn = len(n_arr)
    p_sim = baseline + target_spd
    e_sims = rng.binomial(n_arr[None, :], p_sim, size=(R, Qn))
    detected = 0
    for r in range(R):
        lo, hi = bootstrap_ci(n_arr, e_sims[r], baseline, n_boot, rng)
        if lo > 0 or hi < 0:
            detected += 1
    return detected / R

power_curve = {}
for spd in TARGET_SPDS:
    rng_spd = np.random.default_rng(int(round(spd * 10000)) + 2)  # +2 offset: distinct stream from QS-overall run
    pw = power_at(spd, n_arr, BASELINE_SHARE, R_REPLICATES, N_BOOT_POWER, rng_spd)
    power_curve[spd] = pw
    print(f'target SPD = {spd:+.2f}: power = {pw:.3f}')

## 4. Calibration check (Type-I error)

In [ ]:
type1 = power_curve[0.0]
print(f'Type-I error at target SPD=0: {type1:.3f}  (nominal alpha=0.05)')
if abs(type1 - 0.05) < 0.03:
    print('-> within simulation noise of the nominal rate; pipeline is calibrated.')
else:
    print('-> WARNING: check n_boot / R_REPLICATES, or the bootstrap implementation.')

## 5. Minimum detectable effect (MDE) at 80% power

In [ ]:
spds_sorted = np.array(sorted(power_curve.keys()))
powers_sorted = np.array([power_curve[s] for s in spds_sorted])

if (powers_sorted >= POWER_TARGET).any():
    i = np.argmax(powers_sorted >= POWER_TARGET)
    if i == 0:
        mde_80 = spds_sorted[0]
    else:
        x0, x1 = spds_sorted[i-1], spds_sorted[i]
        y0, y1 = powers_sorted[i-1], powers_sorted[i]
        mde_80 = x0 + (POWER_TARGET - y0) * (x1 - x0) / (y1 - y0)
else:
    mde_80 = float('nan')

power_at_observed = np.interp(OBSERVED_SPD, spds_sorted, powers_sorted)
print(f'MDE at 80% power        : SPD ~= {mde_80:.4f}')
print(f'Observed point estimate : SPD  = {OBSERVED_SPD:+.4f}')
print(f'Power AT the observed point estimate (if it were the true effect): {power_at_observed:.3f}')

## 6. Compare to QS-overall power analysis

Sanity check: the bio-specific MDE/power should be **similar** to the
QS-overall notebook's result (`results/rq1_power_analysis.json`), since
sample size and baseline share are close (591 vs 590 slots, 0.158 vs 0.144
baseline). A large divergence would be a flag to investigate.

In [ ]:
qs_overall_path = find('rq1_power_analysis.json')
with open(qs_overall_path) as f:
    qs_overall = json.load(f)
print(f"QS-overall MDE@80%: {qs_overall['mde_at_80pct_power']:.4f}  vs  bio MDE@80%: {mde_80:.4f}")
print(f"QS-overall power-at-observed: {qs_overall['power_at_observed_spd']:.3f} "
      f"(at SPD={qs_overall['observed_spd']})  vs  bio power-at-observed: {power_at_observed:.3f} "
      f"(at SPD={OBSERVED_SPD})")

## 7. Power curve plot

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(spds_sorted, powers_sorted, 'o-', color='#2e7d32', label='Empirical power (bio-specific list)')
ax.axhline(POWER_TARGET, ls='--', c='#888', lw=1, label='80% power target')
ax.axhline(0.05, ls=':', c='#bbb', lw=1, label='5% (Type-I reference)')
ax.axvline(OBSERVED_SPD, ls='-.', c='#c0392b', lw=1.3,
           label=f'Observed bio-list point estimate ({OBSERVED_SPD:+.3f})')
if not np.isnan(mde_80):
    ax.plot([mde_80], [POWER_TARGET], 'D', color='#c0392b', markersize=7)
    ax.annotate(f'MDE@80% \u2248 {mde_80:.3f}', (mde_80, POWER_TARGET),
                textcoords='offset points', xytext=(8, -14), fontsize=9)
ax.set_xlabel('Target (synthetic) true SPD')
ax.set_ylabel('Power = P(bootstrap CI excludes 0)')
ax.set_title(f'RQ1 statistical power vs. true SPD -- bio-specific elite list\n'
             f'(n={Q} queries, {int(n_arr.sum())} labeled slots)')
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('rq1_power_curve_bio.png', dpi=120)
plt.show()

## 8. Save results

In [ ]:
out = {
    'elite_list': 'QS World University Rankings by Subject 2026: Biological Sciences',
    'baseline_share': BASELINE_SHARE,
    'n_queries': int(Q),
    'n_labeled_slots': int(n_arr.sum()),
    'observed_spd': OBSERVED_SPD,
    'power_curve': {str(k): round(float(v), 4) for k, v in power_curve.items()},
    'type1_error_at_spd0': round(float(type1), 4),
    'mde_at_80pct_power': round(float(mde_80), 4) if not np.isnan(mde_80) else None,
    'power_at_observed_spd': round(float(power_at_observed), 4),
    'compare_qs_overall': {
        'mde_at_80pct_power': qs_overall['mde_at_80pct_power'],
        'power_at_observed_spd': qs_overall['power_at_observed_spd'],
    },
    'config': {
        'n_boot_power': N_BOOT_POWER, 'r_replicates': R_REPLICATES,
        'target_spds': TARGET_SPDS,
    },
}
os.makedirs('results', exist_ok=True)
with open('results/rq1_power_analysis_bio.json', 'w') as f:
    json.dump(out, f, indent=2)
print('saved results/rq1_power_analysis_bio.json')
print(json.dumps(out, indent=2))

## 9. Reading the result

If the MDE/power-at-observed numbers here are close to the QS-overall
notebook's, that confirms the power/MDE conclusion -- not just the point
estimate and significance -- is robust to the choice of elite-institution
list, strengthening the overall RQ1 robustness story in
`rq1_methodology.md` §10.

**Limitations:** identical to `power_analysis.ipynb` §9 (independent
per-query synthetic-effect injection likely makes the MDE a lower bound;
this analysis targets RQ1 only).